## Introduction

After compressing a model (pruning, quantization, sparsification), you want to know: *how much did it actually improve?* fasterbench's `ComparisonReport` answers this by benchmarking both versions and generating a professional report with metric deltas, improvement rankings, and optional radar charts.

This tutorial shows the full workflow: benchmark → compress → benchmark again → generate report.

## Setup

In [ ]:
import torch, torch.nn as nn
from fasterbench import benchmark, Report, ComparisonReport

## Step 1: Benchmark the Original Model

In [ ]:
from torchvision.models import resnet18

model = resnet18(pretrained=True)
x = torch.randn(1, 3, 224, 224)

result_before = benchmark(model, x, metrics=["size", "speed", "compute"])
result_before.summary()

═══ Size ══════════════════════════════════
  Parameters.............. 11.69M
  Model size.............. 44.59 MiB
═══ Speed (cpu) ═══════════════════════════
  Mean latency............ 28.45 ms
  Throughput.............. 35.2 inf/s
═══ Compute ═══════════════════════════════
  MACs.................... 1,819.1 M


## Step 2: Compress the Model

Use any compression technique — pruning, quantization, sparsification:

In [ ]:
# Example: structured pruning with fasterai
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

pruner = Pruner(model, pruning_ratio=0.5, context='local', criteria=large_final,
                example_inputs=x)
pruner.prune_model()

Ignoring output layer: fc
Total ignored layers: 1


## Step 3: Benchmark the Compressed Model

In [ ]:
result_after = benchmark(model, x, metrics=["size", "speed", "compute"])
result_after.summary()

═══ Size ══════════════════════════════════
  Parameters.............. 3.21M
  Model size.............. 12.34 MiB
═══ Speed (cpu) ═══════════════════════════
  Mean latency............ 11.23 ms
  Throughput.............. 89.1 inf/s
═══ Compute ═══════════════════════════════
  MACs.................... 487.3 M


## Step 4: Generate Comparison Report

In [ ]:
report = ComparisonReport(
    result_before, result_after,
    before_name="ResNet-18 (original)",
    after_name="ResNet-18 (50% pruned)",
    title="Pruning Compression Report"
)

report.summary()

              Pruning Compression Report                         

Before: ResNet-18 (original)
After:  ResNet-18 (50% pruned)

═══ Executive Summary ═════════════════════════════════════════
                     Before          After       Change
  Parameters........ 11.69M          3.21M       -72.5% ✓
  Model Size........ 44.59 MiB       12.34 MiB   -72.3% ✓
  Latency (CPU)..... 28.45 ms        11.23 ms    -60.5% ✓
  Throughput (CPU).. 35.2 inf/s      89.1 inf/s  +153.1% ✓
  MACs.............. 1819.1 M        487.3 M     -73.2% ✓

═══ Top Improvements ═════════════════════════════════════════
  1. Throughput (CPU): +153.1%
  2. MACs: -73.2%
  3. Parameters: -72.5%



## Export to HTML or Markdown

Generate shareable reports:

In [ ]:
# HTML report with embedded radar chart
report.to_html("compression_report.html", include_charts=True)

# Markdown report (great for GitHub PRs)
md = report.to_markdown("compression_report.md")
print(md[:200])

# Pruning Compression Report

**Before:** ResNet-18 (original)
**After:** ResNet-18 (50% pruned)

## Summary

| Metric | Before | After | Change |
|--------|--------|-------|--------|


## Single Model Report

`Report` generates a standalone report for a single model:

In [ ]:
single = Report(result_before, model_name="ResNet-18", 
               description="Baseline ImageNet classifier")
single.summary()

# Also supports HTML and Markdown export
single.to_html("resnet18_report.html")

                        ResNet-18                           

Baseline ImageNet classifier

═══ Metrics ═══════════════════════════════════════════════
  Parameters...................... 11.69M
  Model Size...................... 44.59 MiB
  Latency (CPU)................... 28.45 ms
  Throughput (CPU)................ 35.2 inf/s
  MACs............................ 1819.1 M



## Programmatic Access

Access report data as dictionaries for further processing:

In [ ]:
# Get all deltas as a list
for d in report.deltas:
    if d.improved:
        print(f"{d.label}: {d.delta_pct:+.1f}% ({'improved' if d.improved else 'regressed'})")

# Top 3 improvements
for d in report.top_improvements(3):
    print(f"  {d.label}: {d.before:.0f} → {d.after:.0f}")

# Serialize to dict (for JSON, databases, etc.)
data = report.as_dict()

Parameters: -72.5% (improved)
Model Size: -72.3% (improved)
Latency (CPU): -60.5% (improved)
Throughput (CPU): +153.1% (improved)
MACs: -73.2% (improved)
  Throughput (CPU): 35 → 89
  MACs: 1819 → 487
  Parameters: 11690000 → 3210000


---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `Report(result)` | Single model report |
| `ComparisonReport(before, after)` | Before/after comparison |
| `.summary()` | Console output |
| `.to_html(path)` | HTML with radar charts |
| `.to_markdown(path)` | Markdown (great for PRs) |
| `.deltas` | List of `ReportMetricDelta` |
| `.top_improvements(n)` | Top N improvements sorted by impact |

---

## See Also

- [Benchmark](../analysis/benchmark.html) — Unified benchmarking API
- [Visualization](../visualization/plot.html) — Radar plots for visual comparison
- [Report API](../analysis/report.html) — Full API reference